# 03 · Training and comparing the neural models

Phase 1 showed a real but thin mean-reversion signal: the OU baseline is profitable before costs and
loses money after them. Here three neural networks (an MLP, a temporal CNN and a Transformer) are
trained to maximize the **portfolio's Sharpe ratio after 5 bps costs** directly. Their scores go
through the same factor-hedged portfolio layer and backtester as OU.

**Phase 2 rule: only walk-forward Fold 1 is used.** That means training on 2010-2017 and validating
on 2018-2019. The 2020-2023 test years stay untouched until Phase 3, so no modelling choice here can
be fitted to them.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sharpee.backtest.engine import run_backtest
from sharpee.config import load_config, repo_path
from sharpee.evaluation.metrics import summarize
from sharpee.evaluation.walk_forward import make_folds, split
from sharpee.pipeline import load_panel, runs_dir
from sharpee.strategies.ou_strategy import ou_fit, ou_positions
from sharpee.training.dataset import DayBlocks
from sharpee.training.train import predict
from sharpee.training.trials import load_trials
from sharpee.training.tuning import train_config

plt.rcParams.update({"figure.figsize": (9, 3.5), "axes.grid": True, "grid.alpha": 0.3})
FIGURES = repo_path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    """Keep a copy in reports/figures for the write-up."""
    fig.savefig(FIGURES / f"{name}.png", dpi=120, bbox_inches="tight")


cfg = load_config("data", "experiment", "baseline")
runs = runs_dir(cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
panel = load_panel(cfg)
fold = make_folds(cfg["test_years"], cfg["train_start"], cfg["val_years"])[0]
train_idx, val_idx, _ = split(panel, fold, cfg["embargo_days"])  # the test split is deliberately unused

MODELS = ["mlp", "temporal_cnn", "transformer"]
NAMES = {"ou": "OU", "mlp": "MLP", "temporal_cnn": "Temporal CNN", "transformer": "Transformer"}
COLORS = {"ou": "tab:gray", "mlp": "tab:blue", "temporal_cnn": "tab:orange", "transformer": "tab:green"}
print(f"train {panel.dates[train_idx[0]].date()} .. {panel.dates[train_idx[-1]].date()} ({len(train_idx)} days)")
print(f"validate {panel.dates[val_idx[0]].date()} .. {panel.dates[val_idx[-1]].date()} ({len(val_idx)} days), "
      f"30-day embargo before each, device {device}")

## 1. Tuning: every configuration tried

`scripts/train_model.py --tune` and `scripts/run_baseline.py --tune` try a small grid for each
strategy on Fold 1 and log every configuration to `runs/trials.csv`, winners and losers alike. The
spread of these Sharpe ratios is what the Deflated Sharpe Ratio uses in Phase 3: the more
configurations tried, the higher the bar a result must clear.

In [ ]:
trials = load_trials(runs)
trials = trials[trials["fold"] == fold.name].copy()


def describe(row):
    p = json.loads(row.params)
    if row.model == "ou":
        return f"entry {p['entry']}"
    size = {k: v for k, v in p["params"].items() if k in ("hidden", "channels", "d_model")}
    return ", ".join(f"{k} {v}" for k, v in size.items()) + f", lr {p['train']['lr']}"


trials["config"] = trials.apply(describe, axis=1)
trials["strategy"] = trials["model"].map(NAMES)
order = ["ou"] + MODELS
trials = trials.sort_values(["model", "val_sharpe"], key=lambda s: s.map(order.index) if s.name == "model" else -s)

fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{NAMES[m]}: {c}" for m, c in zip(trials["model"], trials["config"])]
ax.barh(labels, trials["val_sharpe"], color=[COLORS[m] for m in trials["model"]])
ax.axvline(0, color="black", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("validation net Sharpe (5 bps)")
ax.set_title(f"All {len(trials)} configurations tried on Fold 1 validation (2018-2019)")
save(fig, "03_tuning_trials")
plt.show()
trials[["strategy", "config", "val_sharpe", "trial_id"]].round(2).reset_index(drop=True)

## 2. Retrain the selected configurations with three seeds

A neural network's result also depends on its random initialization. Retraining each selected
configuration with seeds 0, 1 and 2 shows how much of a difference comes from the model and how much
from luck. Seed reruns don't add configurations, so they are not logged as new trials.

In [ ]:
assert {json.loads((runs / f"best_{m}.json").read_text())["config"]["lookback"] for m in MODELS} == {30}
train_data = DayBlocks(panel, train_idx, 30, device)
val_data = DayBlocks(panel, val_idx, 30, device)

SEEDS = [0, 1, 2]
fitted = {}
for name in MODELS:
    best_cfg = json.loads((runs / f"best_{name}.json").read_text())["config"]
    for seed in SEEDS:
        model, result, _ = train_config(best_cfg, train_data, val_data, cfg["cap"], cfg["cost_bps"],
                                        seed, device, log=lambda _: None)
        fitted[(name, seed)] = {"model": model, "history": pd.DataFrame(result["history"]),
                                "best_epoch": result["best_epoch"], "scores": predict(model, val_data)}
        print(f"{NAMES[name]:13s} seed {seed}: best validation net Sharpe {result['best_val_sharpe']:+.2f} "
              f"at epoch {result['best_epoch']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, name in zip(axes, MODELS):
    for seed in SEEDS:
        h = fitted[(name, seed)]["history"]
        ax.plot(h["epoch"], h["val_sharpe"], color=COLORS[name], alpha=0.4 + 0.2 * seed, label=f"seed {seed}")
        b = fitted[(name, seed)]["best_epoch"]
        ax.scatter([b], [h.loc[h["epoch"] == b, "val_sharpe"]], color=COLORS[name], zorder=3)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(NAMES[name])
    ax.set_xlabel("epoch")
axes[0].set_ylabel("validation net Sharpe")
axes[0].legend()
fig.suptitle("Training curves: validation net Sharpe per epoch (dot = epoch kept by early stopping)")
fig.tight_layout()
save(fig, "03_training_curves")
plt.show()

## 3. Validation comparison, 2018-2019

Every strategy is run through the same backtester at 0, 5, 10 and 20 bps. OU uses its tuned entry
threshold.

Validation days were used both for early stopping and for choosing configurations, so these numbers
are **optimistic** for every strategy. Unbiased numbers come from the untouched test years in Phase 3.
With about 470 days, a single annualized Sharpe estimate has a standard error of roughly 0.7, so
small gaps between strategies are not evidence.

In [ ]:
ou_params = json.loads((runs / "best_ou.json").read_text())["params"]
ou_scores = ou_positions(panel, np.arange(len(panel.dates)), **ou_params)[val_idx]

rows, frames = [], {}


def evaluate(name, seed, scores):
    for cost in cfg["cost_grid"]:
        # OU's +1/-1 positions are taken as they are; model scores are centered
        frame = run_backtest(panel, val_idx, scores, cost_bps=cost, cap=cfg["cap"], center=name != "ou")
        rows.append({"strategy": name, "seed": seed, "cost_bps": cost, **summarize(frame)})
        if cost == cfg["cost_bps"]:
            frames[(name, seed)] = frame


evaluate("ou", 0, ou_scores)
for (name, seed), f in fitted.items():
    evaluate(name, seed, f["scores"])
results = pd.DataFrame(rows)

main = results[results["cost_bps"] == cfg["cost_bps"]]
cols = {"sharpe": "net Sharpe", "gross_sharpe": "gross Sharpe", "ann_return": "ann. return",
        "max_drawdown": "max drawdown", "avg_turnover": "turnover / day", "beta": "beta"}
mean = main.groupby("strategy")[list(cols)].mean().rename(columns=cols)
spread = main.groupby("strategy")[["sharpe"]].std().rename(columns={"sharpe": "net Sharpe, seed sd"})
summary = mean.join(spread).loc[["ou"] + MODELS].rename(index=NAMES)
se = np.sqrt(252 / len(val_idx))
print(f"standard error of one annualized Sharpe over {len(val_idx)} days: about {se:.2f}")

reports = repo_path(cfg["reports_dir"])
results.to_csv(reports / "phase2_validation_runs.csv", index=False)
summary.round(4).to_csv(reports / "phase2_validation_summary.csv")
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(len(summary))
colors = [COLORS[k] for k in ["ou"] + MODELS]
axes[0].bar(x - 0.2, summary["gross Sharpe"], width=0.4, color=colors, alpha=0.45, label="gross")
axes[0].bar(x + 0.2, summary["net Sharpe"], width=0.4, color=colors,
            yerr=summary["net Sharpe, seed sd"].fillna(0), capsize=4, label="net of 5 bps")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xticks(x, summary.index)
axes[0].set_title("Validation Sharpe (light = gross, dark = net; bars = seed sd)")
axes[1].bar(x, summary["turnover / day"], color=colors)
axes[1].set_xticks(x, summary.index)
axes[1].set_title("Average daily turnover (share of the book traded)")
fig.tight_layout()
save(fig, "03_validation_comparison")
plt.show()

In [ ]:
by_cost = results.groupby(["strategy", "cost_bps"])["sharpe"].mean().unstack(0)[["ou"] + MODELS]
ax = by_cost.rename(columns=NAMES).plot(marker="o", color=[COLORS[k] for k in ["ou"] + MODELS])
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("cost per unit traded (bps)")
ax.set_ylabel("validation net Sharpe")
ax.set_title("Cost sensitivity on validation (seed mean)")
save(ax.figure, "03_cost_sensitivity")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
for (name, seed), frame in frames.items():
    eq = (1 + frame["net"]).cumprod()
    ax.plot(eq.index, eq.values, color=COLORS[name], lw=2 if name == "ou" else 1,
            alpha=1 if name == "ou" else 0.35 + 0.2 * seed, label=NAMES[name] if seed == 0 else None)
ax.axhline(1, color="black", lw=0.8)
ax.set_ylabel("growth of 1")
ax.set_title("Validation equity, net of 5 bps (one line per seed)")
ax.legend()
save(fig, "03_validation_equity")
plt.show()

## 4. What did the networks learn?

Each validation stock-day gets an OU s-score (how stretched its cumulative residual is) and a
network score. Within each day the network scores are standardized across stocks, then averaged by
s-score bucket. A network that rediscovered mean reversion would show a downward slope: it buys
residuals that fell (negative s) and sells ones that rose.

In [ ]:
s_val, _, ok = ou_fit(panel.windows(val_idx, cfg["window"]))
ok &= panel.tradable[val_idx]
edges = np.arange(-3, 3.01, 0.5)
mid = (edges[:-1] + edges[1:]) / 2

fig, ax = plt.subplots(figsize=(9, 4))
corr = {}
for name in MODELS:
    sc = fitted[(name, 0)]["scores"].astype(np.float64)
    sc = np.where(panel.tradable[val_idx], sc, np.nan)
    z = (sc - np.nanmean(sc, axis=1, keepdims=True)) / np.nanstd(sc, axis=1, keepdims=True)
    b = np.digitize(s_val[ok], edges) - 1
    keep = (b >= 0) & (b < len(mid))
    curve = pd.Series(z[ok][keep]).groupby(b[keep]).mean()
    ax.plot(mid[curve.index], curve.values, marker="o", color=COLORS[name], label=NAMES[name])
    daily = [pd.Series(z[d][ok[d]]).corr(pd.Series(s_val[d][ok[d]]), method="spearman") for d in range(len(val_idx))]
    corr[NAMES[name]] = np.nanmean(daily)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("OU s-score bucket")
ax.set_ylabel("mean standardized network score")
ax.set_title("Network score vs OU s-score on validation (seed 0)")
ax.legend()
save(fig, "03_score_vs_sscore")
plt.show()
pd.Series(corr, name="mean daily rank correlation with s-score").round(3).to_frame()